In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02. DFMGAN Training\n",
    "Training the Discriminator-Filtered Monte Carlo GAN"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.append(os.path.abspath('..'))\n",
    "\n",
    "import torch\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from src.pipeline.data_fetcher import DataFetcher\n",
    "from src.pipeline.dfmgan.generator import DFMGANConfig, ConditionalMCGenerator, TimeGANDiscriminator\n",
    "from src.pipeline.dfmgan.trainer import DFMGANTrainer\n",
    "\n",
    "%matplotlib inline\n",
    "\n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f\"[INFO] Using device: {device}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Load data\n",
    "fetcher = DataFetcher()\n",
    "tickers = ['AAPL', 'MSFT', 'GOOGL']\n",
    "\n",
    "try:\n",
    "    data = fetcher.get_historical_data(tickers, period='5y')\n",
    "\n",
    "    if data is None or data.empty:\n",
    "        raise ValueError(\"No data fetched\")\n",
    "\n",
    "    data = data[tickers].dropna()\n",
    "    print(f\"[INFO] Data loaded: {data.shape}\")\n",
    "\n",
    "except Exception as e:\n",
    "    print(f\"[ERROR] Data loading failed: {e}\")\n",
    "    data = pd.DataFrame()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Prepare sequences\n",
    "seq_len = 60\n",
    "\n",
    "if not data.empty and len(data) > seq_len:\n",
    "    values = data.values\n",
    "\n",
    "    sequences = []\n",
    "    for i in range(len(values) - seq_len):\n",
    "        sequences.append(values[i:i+seq_len])\n",
    "\n",
    "    real_data = torch.tensor(np.array(sequences), dtype=torch.float32).to(device)\n",
    "\n",
    "    print(f\"[INFO] Real data shape: {real_data.shape}\")\n",
    "else:\n",
    "    real_data = None\n",
    "    print(\"[ERROR] Not enough data for sequence generation\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Initialize models\n",
    "if real_data is not None:\n",
    "    config = DFMGANConfig(\n",
    "        n_assets=len(tickers),\n",
    "        seq_len=seq_len,\n",
    "        latent_dim=64,\n",
    "        hidden_dim=128,\n",
    "        n_simulations=1000\n",
    "    )\n",
    "\n",
    "    generator = ConditionalMCGenerator(config).to(device)\n",
    "    discriminator = TimeGANDiscriminator(config).to(device)\n",
    "\n",
    "    print(f\"Generator parameters: {sum(p.numel() for p in generator.parameters()):,}\")\n",
    "    print(f\"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}\")\n",
    "else:\n",
    "    print(\"[ERROR] Model initialization skipped\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Train model\n",
    "if real_data is not None:\n",
    "    trainer = DFMGANTrainer(generator, discriminator, config, device=device)\n",
    "\n",
    "    history = trainer.train(\n",
    "        real_data=real_data,\n",
    "        epochs=50,\n",
    "        batch_size=32\n",
    "    )\n",
    "else:\n",
    "    history = None\n",
    "    print(\"[ERROR] Training skipped\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Plot training history\n",
    "if history is not None:\n",
    "    fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "\n",
    "    axes[0].plot(history.get('d_loss', []), label='Discriminator')\n",
    "    axes[0].plot(history.get('g_loss', []), label='Generator')\n",
    "    axes[0].set_title('Training Loss')\n",
    "    axes[0].legend()\n",
    "\n",
    "    axes[1].plot(history.get('d_real_acc', []), label='Real Acc')\n",
    "    axes[1].plot(history.get('d_fake_acc', []), label='Fake Acc')\n",
    "    axes[1].set_title('Discriminator Accuracy')\n",
    "    axes[1].legend()\n",
    "\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"[WARNING] No training history\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Generate sample paths\n",
    "if real_data is not None:\n",
    "    with torch.no_grad():\n",
    "        sample_hist = real_data[:1, :30, :]\n",
    "        generated = generator(sample_hist, n_paths=100)\n",
    "\n",
    "    print(f\"[INFO] Generated paths shape: {generated.shape}\")\n",
    "else:\n",
    "    generated = None\n",
    "    print(\"[ERROR] Generation skipped\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Plot generated paths\n",
    "if generated is not None:\n",
    "    plt.figure(figsize=(12, 6))\n",
    "\n",
    "    real_plot = real_data[0, :, 0].cpu().numpy()\n",
    "    plt.plot(range(len(real_plot)), real_plot, 'k--', label='Real', linewidth=2)\n",
    "\n",
    "    gen_plot = generated[:10, :, 0].cpu().numpy()\n",
    "    for i in range(10):\n",
    "        plt.plot(range(len(gen_plot[i])), gen_plot[i], alpha=0.3)\n",
    "\n",
    "    plt.xlabel('Time Steps')\n",
    "    plt.ylabel('Price')\n",
    "    plt.title('Generated vs Real')\n",
    "    plt.legend()\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"[WARNING] No generated data\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Save models\n",
    "if real_data is not None:\n",
    "    os.makedirs('../data/models', exist_ok=True)\n",
    "    trainer.save_models('../data/models/dfmgan.pth')\n",
    "    print(\"[INFO] Models saved successfully\")\n",
    "else:\n",
    "    print(\"[ERROR] Model saving skipped\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
